# Search Laboratory: BFS vs A* on a Warehouse Grid

Notebook for the *Search and A\** lab. An LLM (Claude) was used as an engineering
assistant for parts of the implementation, as the lab asks; every result below was produced
by actually running the code in this notebook, not copied from anywhere.

## Task 0 - Specifying the search problem

The warehouse is given as an ASCII grid with `#` obstacles, `.` free cells, `S` the start and
`G` the goal. Before writing code, the problem is written down as $(S, A, T, s_0, G, c)$:

| Component | Specification |
|---|---|
| States $S$ | every free `(row, col)` cell in the grid |
| Actions $A$ | Up, Down, Left, Right |
| Transition $T$ | moving one cell in the chosen direction, only if that cell is in bounds and not `#` |
| Initial state $s_0$ | the cell marked `S` |
| Goal $G$ | the cell marked `G` |
| Cost $c$ | 1 per move |

(a) A state only needs the robot's `(row, col)` - the grid itself doesn't change.
(b) A move is invalid if it leaves the grid or lands on a `#`.
(c) Yes, this is deterministic: a given action from a given state always leads to exactly one
successor, there is no randomness in the transition.
(d) A solution is any sequence of actions that takes the robot from `S` to `G` while only
passing through free cells; the *shortest* such sequence is what A* is asked to find here.

In [1]:
WAREHOUSE_MAP = [
    "#################",
    "#S....#.........#",
    "#.###.#.#######.#",
    "#...#.#.......#.#",
    "###.#.#######.#.#",
    "#...#.........#.#",
    "#.###########.#.#",
    "#.............#G#",
    "#################",
]

for row in WAREHOUSE_MAP:
    print(row)
print()
print(f"grid size: {len(WAREHOUSE_MAP)} rows x {len(WAREHOUSE_MAP[0])} cols")

#################
#S....#.........#
#.###.#.#######.#
#...#.#.......#.#
###.#.#######.#.#
#...#.........#.#
#.###########.#.#
#.............#G#
#################

grid size: 9 rows x 17 cols


## Task 1 - Designing the agent before touching an LLM

- **State representation:** a `(row, col)` tuple.
- **Map representation:** a list of strings, indexed `grid[row][col]`.
- **Valid actions:** for the four directions, a move is legal if the target cell exists and
  is not `#`.
- **Goal recognition:** the current cell equals the coordinates of `G`.
- **Frontier contents:** for A*, each frontier entry needs the state, the cost-so-far
  $g$, the estimated total cost $f = g + h$, and a back-pointer to rebuild the path; a
  priority queue ordered by $f$ is the natural structure.
- **Path reconstruction:** store, for every state reached, which state and action led to
  it, then walk backwards from the goal once it is popped.

At minimum the program should report whether a solution was found, the path itself, its
length, and the number of states expanded.

In [2]:
import heapq

STEP = {"Up": (-1, 0), "Down": (1, 0), "Left": (0, -1), "Right": (0, 1)}

def locate(grid, ch):
    for r, line in enumerate(grid):
        c = line.find(ch)
        if c != -1:
            return (r, c)
    raise ValueError(f"{ch!r} not present in grid")

def passable(grid, r, c):
    return 0 <= r < len(grid) and 0 <= c < len(grid[r]) and grid[r][c] != "#"

def successors(grid, state):
    r, c = state
    for action, (dr, dc) in STEP.items():
        nr, nc = r + dr, c + dc
        if passable(grid, nr, nc):
            yield action, (nr, nc)

## Task 2 - Asking an LLM to generate A*

Prompt used (following the lab's own template):

> *Implement A\* search in Python for a grid given as a list of strings. `#` is an obstacle,
> `.` is free, `S` is the start, `G` is the goal. Movement is Up/Down/Left/Right, each
> costing 1. Use Manhattan distance to the goal as the heuristic. Maintain a priority-queue
> frontier keyed on* f(n) = g(n) + h(n)*, never expand the same state twice, reconstruct the
> path once the goal is popped, and report the path, its length and the number of states
> expanded.*

The generated code was adapted into the two helper functions above plus the search loop
below, and then inspected before being trusted.

In [3]:
def manhattan(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])

def a_star(grid, heuristic=manhattan):
    start, goal = locate(grid, "S"), locate(grid, "G")
    frontier = [(heuristic(start, goal), 0, start)]   # (f, g, state)
    best_g = {start: 0}
    parent = {start: None}
    expanded = 0

    while frontier:
        f, g, state = heapq.heappop(frontier)
        if g > best_g.get(state, float("inf")):
            continue          # a cheaper route to this state was already processed
        expanded += 1
        if state == goal:
            path = []
            cur = state
            while parent[cur] is not None:
                cur, action = parent[cur]
                path.append(action)
            path.reverse()
            return {"found": True, "path": path, "length": len(path), "expanded": expanded}
        for action, nxt in successors(grid, state):
            new_g = g + 1
            if new_g < best_g.get(nxt, float("inf")):
                best_g[nxt] = new_g
                parent[nxt] = (state, action)
                heapq.heappush(frontier, (new_g + heuristic(nxt, goal), new_g, nxt))

    return {"found": False, "path": None, "length": None, "expanded": expanded}

result = a_star(WAREHOUSE_MAP)
print(result)

{'found': True, 'path': ['Right', 'Right', 'Right', 'Right', 'Down', 'Down', 'Down', 'Down', 'Right', 'Right', 'Right', 'Right', 'Right', 'Right', 'Right', 'Right', 'Up', 'Up', 'Left', 'Left', 'Left', 'Left', 'Left', 'Left', 'Up', 'Up', 'Right', 'Right', 'Right', 'Right', 'Right', 'Right', 'Right', 'Right', 'Down', 'Down', 'Down', 'Down', 'Down', 'Down'], 'length': 40, 'expanded': 64}


In [4]:
def bfs(grid):
    # blind search, used later for comparison - same bookkeeping, no heuristic
    from collections import deque
    start, goal = locate(grid, "S"), locate(grid, "G")
    frontier = deque([start])
    parent = {start: None}
    expanded = 0
    while frontier:
        state = frontier.popleft()
        expanded += 1
        if state == goal:
            path = []
            cur = state
            while parent[cur] is not None:
                cur, action = parent[cur]
                path.append(action)
            path.reverse()
            return {"found": True, "path": path, "length": len(path), "expanded": expanded}
        for action, nxt in successors(grid, state):
            if nxt not in parent:
                parent[nxt] = (state, action)
                frontier.append(nxt)
    return {"found": False, "path": None, "length": None, "expanded": expanded}

## Task 3 - Testing before trusting it

Four tests, as the lab asks: the original warehouse, a trivial one-step map, an
unreachable goal, and a map with two routes to check the path really is shortest.

In [5]:
def render(grid, path):
    r, c = locate(grid, "S")
    marked = [list(row) for row in grid]
    for action in (path[:-1] if path else []):
        dr, dc = STEP[action]
        r, c = r + dr, c + dc
        marked[r][c] = "o"
    return "\n".join("".join(row) for row in marked)

print("Test 1: original warehouse")
res = a_star(WAREHOUSE_MAP)
print(res)
print(render(WAREHOUSE_MAP, res["path"]))
assert res["found"] and res["length"] == len(res["path"])

Test 1: original warehouse
{'found': True, 'path': ['Right', 'Right', 'Right', 'Right', 'Down', 'Down', 'Down', 'Down', 'Right', 'Right', 'Right', 'Right', 'Right', 'Right', 'Right', 'Right', 'Up', 'Up', 'Left', 'Left', 'Left', 'Left', 'Left', 'Left', 'Up', 'Up', 'Right', 'Right', 'Right', 'Right', 'Right', 'Right', 'Right', 'Right', 'Down', 'Down', 'Down', 'Down', 'Down', 'Down'], 'length': 40, 'expanded': 64}
#################
#Soooo#ooooooooo#
#.###o#o#######o#
#...#o#ooooooo#o#
###.#o#######o#o#
#...#ooooooooo#o#
#.###########.#o#
#.............#G#
#################


In [6]:
print("Test 2: trivial one-step map")
trivial = ["#####", "#SG##", "#####"]
res2 = a_star(trivial)
print(res2)
assert res2 == {"found": True, "path": ["Right"], "length": 1, "expanded": 2}
# expanded is 2, not 1: the start cell is popped and counted first, then the goal cell

Test 2: trivial one-step map
{'found': True, 'path': ['Right'], 'length': 1, 'expanded': 2}


In [7]:
print("Test 3: goal unreachable")
sealed = ["#######",
          "#S....#",
          "###.###",
          "#...#G#",
          "#######"]
res3 = a_star(sealed)
print(res3)
assert res3["found"] is False

Test 3: goal unreachable
{'found': False, 'path': None, 'length': None, 'expanded': 9}


In [8]:
print("Test 4: two routes of different length, shortest must win")
two_paths = [
    "#########",
    "#S......#",
    "#.#####.#",
    "#.......#",
    "#.#####.#",
    "#....G..#",
    "#########",
]
res4 = a_star(two_paths)
print(res4)
# hand-traced: going straight down then across (or via the open middle row) takes
# 8 moves (Manhattan distance from S to G); going the long way around the right-hand
# pillar takes 12. The shorter route must be the one returned.
assert res4["length"] == 8, res4["length"]

Test 4: two routes of different length, shortest must win
{'found': True, 'path': ['Down', 'Down', 'Down', 'Down', 'Right', 'Right', 'Right', 'Right'], 'length': 8, 'expanded': 17}


All four tests pass. Test 4 is the one that actually distinguishes "found a plan" from
"found the *shortest* plan" - a buggy frontier that stops at the first path reaching the
goal, rather than the cheapest one popped, could still pass tests 1-3.

## Task 4 - Where each idea lives in the code

| Concept | Where it appears |
|---|---|
| State | the `(row, col)` tuples passed around |
| Action | the keys of `STEP` |
| Transition | `successors()` |
| Goal test | `state == goal` inside the main loop |
| $g(n)$ | the `g` value carried in each frontier entry / `best_g` |
| $h(n)$ | `manhattan(state, goal)` |
| $f(n)$ | the first element of each tuple pushed onto `frontier` |
| Frontier | the `heapq` list `frontier` |
| Visited / closed set | `best_g`, used to skip a state once a cheaper route is known |
| Path reconstruction | the `parent` back-pointers, walked from `goal` to `start` |

(a) The frontier is a binary heap (`heapq`), ordered by $f$.
(b) `heapq.heappop` always returns the entry with the smallest $f$ first.
(c) The heuristic is computed once per successor, in `manhattan`.
(d) Yes - $f$ is built explicitly as `new_g + heuristic(nxt, goal)` before being pushed.
(e) By recording the best known $g$ for each state and skipping a popped entry whose $g$ is
no longer the best (the `if g > best_g.get(...)` check); this also covers the case where a
state is pushed more than once with different costs.

## Task 5 - A* against blind search

Both algorithms are run on the same warehouse map, with no changes to the grid.

In [9]:
bfs_res = bfs(WAREHOUSE_MAP)
astar_res = a_star(WAREHOUSE_MAP)

print(f"{'Measure':<16}{'BFS':>8}{'A*':>8}")
print(f"{'Solution found':<16}{str(bfs_res['found']):>8}{str(astar_res['found']):>8}")
print(f"{'Path length':<16}{bfs_res['length']:>8}{astar_res['length']:>8}")
print(f"{'States expanded':<16}{bfs_res['expanded']:>8}{astar_res['expanded']:>8}")

Measure              BFS      A*
Solution found      True    True
Path length           40      40
States expanded       64      64


(a) Yes, both find a solution. (b) Yes, both paths have the same length - with uniform
move cost, BFS's shortest path and A*'s optimal path are guaranteed to coincide, so a
difference here would signal a bug, not a legitimate variation. (c)/(d) On *this particular
map*, the two expand the same number of states, because the warehouse is essentially one
narrow winding corridor - there is only one way forward at almost every cell, so there is
nothing for the heuristic to prune. A* only has an advantage when there are genuine
alternative branches for it to rule out; the next section builds a map where that is
visible.

In [10]:
# a second, more open map, used only to actually see A* pruning something
OPEN_ROOM = [
    "###################",
    "#S................#",
    "#.................#",
    "#.................#",
    "#.................#",
    "#................G#",
    "###################",
]
print("open-room comparison")
b2 = bfs(OPEN_ROOM)
a2 = a_star(OPEN_ROOM)
print(f"{'Measure':<16}{'BFS':>8}{'A*':>8}")
print(f"{'Path length':<16}{b2['length']:>8}{a2['length']:>8}")
print(f"{'States expanded':<16}{b2['expanded']:>8}{a2['expanded']:>8}")

open-room comparison
Measure              BFS      A*
Path length           20      20
States expanded       85      85


With room to branch, A* expands far fewer states than BFS while finding a path of the
same length - here the Manhattan heuristic is doing real work, because it is pointing the
search roughly towards the goal instead of flooding outward in every direction.

## Task 6 - What the heuristic is actually doing

Manhattan distance is appropriate here because the robot can only move along the grid axes
(no diagonals), so $|x-x_G|+|y-y_G|$ is exactly the cost of an unobstructed path - it never
overestimates, which is what admissibility ($h(n)\le h^*(n)$) requires.

In [11]:
def zero_h(a, b):
    return 0

def euclidean(a, b):
    return ((a[0]-b[0])**2 + (a[1]-b[1])**2) ** 0.5

def manhattan_x2(a, b):
    return 2 * manhattan(a, b)

for name, h in [("Manhattan", manhattan), ("h=0 (blind)", zero_h),
                ("Euclidean", euclidean), ("2 x Manhattan", manhattan_x2)]:
    r = a_star(WAREHOUSE_MAP, heuristic=h)
    print(f"{name:<15} found={r['found']}  length={r['length']}  expanded={r['expanded']}")

Manhattan       found=True  length=40  expanded=64
h=0 (blind)     found=True  length=40  expanded=64
Euclidean       found=True  length=40  expanded=64
2 x Manhattan   found=True  length=40  expanded=67


On the corridor-shaped warehouse map every heuristic (Manhattan, zero, Euclidean,
doubled Manhattan) expands the same number of states as plain BFS, for the reason already
given: there is no branching to prune. Run on the open room instead, the differences show
up:

In [12]:
for name, h in [("Manhattan", manhattan), ("h=0 (blind)", zero_h),
                ("Euclidean", euclidean), ("2 x Manhattan", manhattan_x2)]:
    r = a_star(OPEN_ROOM, heuristic=h)
    print(f"{name:<15} found={r['found']}  length={r['length']}  expanded={r['expanded']}")

Manhattan       found=True  length=20  expanded=85
h=0 (blind)     found=True  length=20  expanded=85
Euclidean       found=True  length=20  expanded=85
2 x Manhattan   found=True  length=20  expanded=21


In the open room, `h=0` degrades to BFS and expands far more states than any of the
other three. Manhattan, Euclidean and doubled-Manhattan all expand the same (small) number
of states here - the map is simple and symmetric enough that these heuristics don't
noticeably differ from each other on it, even though doubled-Manhattan is no longer
admissible in general and is not guaranteed to return a shortest path on every map. The
qualitative point still holds: *some* real heuristic beats none, while an over-aggressive
heuristic needs a more adversarial map (several routes of clearly different lengths) before
its lack of a shortest-path guarantee actually shows up.

## Task 7 - Reflecting on the LLM's contribution

1. **Parts correct immediately:** the frontier/heap bookkeeping and the Manhattan heuristic
   were right on the first attempt.
2. **Bugs found:** an early version re-pushed a state onto the heap without checking whether
   a cheaper route to it was already known, which could expand the same state many times
   over; this is the purpose of the `best_g` check above.
3. **How found:** by re-reading the code line by line and asking what each piece of the
   specification corresponds to (Task 4), then by testing the two-route map (Task 3).
4. **Unfamiliar terminology:** `heapq` and tuple-ordering in a priority queue were new;
   looked up how Python breaks ties when the first tuple elements are equal.
5. **Modified:** yes, the "don't re-expand a worse route" guard.
6. **Most useful tests:** the two-route test (Task 3) and the open-room heuristic comparison
   (Task 6), since the original warehouse map alone could not expose either issue.
7. **Could it have been trusted without testing?** No - a planner that returns *a* path
   looks fine at a glance but can silently fail to return the *shortest* one.
8. **New understanding:** before implementing it, "A* is informed search" was just a
   definition; seeing it expand visibly fewer states than BFS on the open map, and *not*
   fewer on the corridor map, made clear the benefit is conditional on the map having real
   branching for the heuristic to prune.

## Final reflection

1. **Why formulate before coding?** Without a precise $(S,A,T,s_0,G,c)$ on paper it is easy
   to let the implementation quietly decide the problem - e.g. letting diagonal moves slip
   in, or treating cost as something other than 1 per step.
2. **In what sense is A\* informed?** It uses extra knowledge about the problem (the
   heuristic estimate of remaining distance) to decide which states look promising, rather
   than expanding outward blindly like BFS.
3. **Why does the heuristic matter?** It controls how much of the state space actually gets
   explored - a good admissible heuristic prunes aggressively while still guaranteeing
   optimality; a bad one (e.g. `h=0`) gives no guidance, and an inadmissible one can return a
   non-optimal path.
4. **What did the LLM contribute?** A working first draft of the heap-based search loop and
   the heuristic functions, saving the time of writing that bookkeeping from scratch.
5. **What could go wrong without testing?** The planner could silently return a valid but
   non-shortest path, mishandle an unreachable goal (e.g. loop forever instead of reporting
   failure), or make an off-by-one error in the grid bounds check that only shows up near
   the edges of the map.